# M3 — Priority (direct predictor)

Predicts `priority` directly from the summary plus `plan_tier`, `tenure` and `channel`. Evaluated on a **chronological split** (and, for reference, a random split). Reports ordinal extras (within-1-level, quadratic kappa) because priority is ordered. Note: this is the one model that keeps `plan_tier`/`tenure`.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
search = os.getcwd()
while search != os.path.dirname(search) and not os.path.exists(os.path.join(search, "case_pipeline.py")):
    search = os.path.dirname(search)
sys.path.insert(0, search)
import numpy as np, pandas as pd
import case_pipeline as P   # shared loading / splits / features / evaluation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

df      = P.load_clean()
FEATURES = ["case_summary", "plan_tier", "channel", "customer_tenure_months"]
TARGET   = "priority"
classes  = P.PRIORITIES                       # ordered: Low < Medium < High < Urgent

def build_model():
    # Summary (stop-words removed, 5k features) + one-hot plan/channel + scaled tenure.
    steps = [("summary", TfidfVectorizer(max_features=5000, stop_words="english",
                                         ngram_range=(1, 2), min_df=2, max_df=0.95), "case_summary"),
             P.t_cat(["plan_tier", "channel"]),
             P.t_num(["customer_tenure_months"])]
    return P.make_clf(steps)

train, _, test = P.time_split(df)             # train on early months, test on later ones
model = build_model().fit(train[FEATURES], train[TARGET])
pred  = model.predict(test[FEATURES])

baseline = df[TARGET].value_counts(normalize=True).iloc[0]
metrics  = P.evaluate(test[TARGET], pred, classes, baseline=baseline)
metrics.update(P.ordinal_extras(test[TARGET], pred, classes))

# Reference: the same model on a random stratified split (shows how a random split flatters).
r_train, r_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df[TARGET])
r_pred = build_model().fit(r_train[FEATURES], r_train[TARGET]).predict(r_test[FEATURES])
metrics["macro_F1_random_split"] = round(float(P.f1_score(r_test[TARGET], r_pred, average="macro")), 3)
metrics

In [ ]:
proba = model.predict_proba(test[FEATURES])
heldout = test[["case_id", "case_summary", "plan_tier", "channel", "customer_tenure_months", TARGET]].copy().reset_index(drop=True)
heldout["predicted"]  = pred
heldout["confidence"] = proba.max(axis=1).round(3)

deployed = build_model().fit(df[FEATURES], df[TARGET])
P.save_bundle("m3_priority_direct", {
    "model": deployed, "target": TARGET, "features": FEATURES, "classes": classes, "kind": "multiclass",
    "label": "Priority — direct", "metrics": metrics, "test": heldout,
    "report": P.class_report(test[TARGET], pred, classes),
    "confusion": P.conf_matrix(test[TARGET], pred, classes),
    "meta": {"contributor": "C", "approach": "direct priority predictor",
             "split": "time (+random ref)", "priorities": classes,
             "inputs": "summary + plan_tier + tenure + channel"}})